In [ ]:
from document_ia_schemas.avis_imposition import AvisImpositionModel
import pandas as pd
from typing import List, Optional, Tuple
import requests
from io import BytesIO
from PIL import Image
import fitz
import os
from pydantic import BaseModel, Field
import numpy as np
import cv2
from pylibdmtx.pylibdmtx import decode
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import gc
import dotenv
import boto3
import json
from botocore.exceptions import ClientError
import logging
import time
from functools import lru_cache
from pathlib import Path
from urllib.parse import quote
from document_ia_evals.utils.label_studio import create_task

from tdd.doc import TwoDDoc

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Load environment once at module level
dotenv.load_dotenv('../.env', override=True)

# Constants
DEFAULT_CSV_PATH = "/home/erwan/Desktop/clients/MinistereTransitionEcologique/document-ia-evals/datasets/2d-doc/tax-notices.csv"
DEFAULT_TIMEOUT = 5000
CHUNK_SIZE = 100
MAX_WORKERS = 10

# List of Pydantic fields that require float conversion
FLOAT_FIELDS = [
    "revenu_fiscal_reference", 
    "nombre_parts",
    "revenu_brut_global",
    "revenu_imposable",
    "impot_revenu_net_avant_corrections",
    "montant_impot"
]

@lru_cache(maxsize=1)
def _get_session_config():
    """Cache session configuration to avoid repeated environment lookups."""
    return {
        'jsessionid': os.getenv("DOSSIER_FACILE_JSESSIONID"),
        's3_endpoint': os.getenv('S3_ENDPOINT'),
        's3_access_key': os.getenv('S3_ACCESS_KEY'),
        's3_secret_key': os.getenv('S3_SECRET_KEY'),
        's3_region': os.getenv('S3_REGION'),
        's3_bucket': os.getenv('S3_BUCKET_NAME')
    }

def load_tax_notice_dataset(csv_path: str = DEFAULT_CSV_PATH) -> List[str]:
    """Load tax notice file IDs from CSV, filtering out invalid entries."""
    if not Path(csv_path).exists():
        raise FileNotFoundError(f"Dataset not found at {csv_path}")
    
    df = pd.read_csv(csv_path, usecols=["file_id"])
    ids = df["file_id"].dropna().astype(str).str.strip()
    valid_ids = [s for s in ids.tolist() if s]
    logger.info(f"Loaded {len(valid_ids)} valid file IDs from {csv_path}")
    return valid_ids


def get_tax_notice_data(file_id: str) -> Tuple[Image.Image, bytes, str]:
    """
    Fetch a file from DossierFacile.
    """
    config = _get_session_config()
    cookies = {'JSESSIONID': config['jsessionid']}
    headers = {'user-agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'}
    
    encoded_id = quote(file_id) 
    url = f'https://bo.dossierfacile.fr/files/{encoded_id}'
    
    try:
        response = requests.get(url, cookies=cookies, headers=headers, timeout=30)
        response.raise_for_status()
    except requests.RequestException as e:
        logger.error(f"Failed to fetch file {file_id}: {e}")
        raise
    
    content_type = response.headers.get('content-type', '').lower()
    raw_content = response.content
    
    # Convert to image for 2D-DOC processing
    if 'pdf' in content_type:
        try:
            doc = fitz.open(stream=raw_content, filetype='pdf')
            page = doc.load_page(0)
            # Render page at 200 DPI for better DataMatrix reading
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2)) 
            img = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
            doc.close()
            del pix, page, doc
            gc.collect()
            return img, raw_content, content_type
        except Exception as e:
            logger.error(f"Failed to convert PDF to image for {file_id}: {e}")
            raise
    else:
        return None, None, content_type
    
    # For image files
    try:
        img = Image.open(BytesIO(raw_content))
        img.load()
        return img, raw_content, content_type
    except Exception as e:
        logger.error(f"Failed to open image for {file_id}: {e}")
        raise

def extract_tax_notice(pil_image: Image.Image, timeout: int = DEFAULT_TIMEOUT) -> Optional[AvisImpositionModel]:
    """
    Extract tax notice information with improved preprocessing and debugging.
    """
    # 1. Convert to grayscale
    img_array = np.array(pil_image)
    if len(img_array.shape) == 3:
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_array

    # 2. Try decoding raw grayscale
    res = decode(gray, timeout=timeout)

    # 3. If failed, try preprocessing (Thresholding + Upscaling)
    if not res:
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        res = decode(thresh, timeout=timeout)
        
        if not res:
            height, width = thresh.shape
            zoomed = cv2.resize(thresh, (width * 2, height * 2), interpolation=cv2.INTER_CUBIC)
            res = decode(zoomed, timeout=timeout)
            del zoomed

    if not res:
        logger.warning("❌ No DataMatrix found in image (Check image quality or scan resolution)")
        return None
    
    logger.info(f"✅ Found {len(res)} DataMatrix code(s)")

    # 4. Parse 2D-DOC
    doc = None
    for potential_doc in res:
        try:
            raw_data = potential_doc.data.decode('latin-1')
            doc = TwoDDoc.from_code(raw_data)
        except Exception as e:
            logger.debug(f"Failed to parse potential 2D-DOC: {e}")
            continue
    
    if doc is None:
        logger.warning("❌ DataMatrix found, but not a valid 2D-DOC")
        return None

    return _parse_2d_doc_to_extract(doc)

def _parse_2d_doc_to_extract(doc: TwoDDoc) -> Optional[AvisImpositionModel]:
    """Parse a TwoDDoc object into AvisImpositionModel."""
    
    # 1. FIX: Map 2D-DOC Label to Pydantic ALIAS (The required dictionary key)
    field_mapping = {
        "Année des revenus": "Année des revenus",
        "Revenu fiscal de référence": "Revenu fiscal de référence",
        "Référence d'avis d'impôt": "Référence d'avis d'impôt",
        "Nombre de parts": "Nombre de parts",
        "Date de mise en recouvrement": "Date de mise en recouvrement",
        "Revenu brut global": "Revenu brut global",
        "Revenu imposable": "Revenu imposable",
        "Impôt sur le revenu net avant corrections": "Impôt sur le revenu net avant corrections",
        "Montant de l'impôt": "Montant de l'impôt",
        "Déclarant 1": "Nom du déclarant 1", 
        "Numéro fiscal du déclarant 1": "Numéro fiscal du déclarant 1",
        "Déclarant 2": "Nom du déclarant 2", 
        "Numéro fiscal du déclarant 2": "Numéro fiscal du déclarant 2",
    }
    
    data = {}
    
    for item in doc.message.dataset:
        # Normalize incoming key (handling curly apostrophes/quotes)
        field_name = item.definition.name.replace("’", "'").strip()
        
        if field_name not in field_mapping:
            continue
            
        # The key for the 'data' dictionary is now the Pydantic ALIAS
        pydantic_alias = field_mapping[field_name]
        raw_value = str(item.value)
        
        # --- Determine the internal Pydantic field name for type checking ---
        # This is required to know if we need to convert to float
        # We must reverse-lookup the field name from the ALIAS.
        # We can use the model schema for this, but for simplicity, we'll use a hardcoded list.
        
        internal_field_name = None
        if pydantic_alias == "Revenu fiscal de référence":
            internal_field_name = "revenu_fiscal_reference"
        elif pydantic_alias == "Nombre de parts":
            internal_field_name = "nombre_parts"
        # Add other float fields here:
        elif pydantic_alias in ["Revenu brut global", "Revenu imposable", "Montant de l'impôt"]:
             internal_field_name = "is_float_field" # Placeholder to trigger float conversion
        # --- End internal field determination ---
        

        # Handle float conversion for numeric fields
        if internal_field_name in FLOAT_FIELDS or internal_field_name == "is_float_field":
            try:
                # Clean value: remove spaces, replace comma with dot
                cleaned_value = raw_value.replace(" ", "").replace(",", ".")
                value = float(cleaned_value)
            except ValueError as e:
                logger.warning(f"Failed to convert '{raw_value}' to float for field {pydantic_alias}: {e}. Skipping.")
                continue
        else:
            value = raw_value # Keep as string for names, references, etc.

        # The key stored in 'data' is the ALIAS
        data[pydantic_alias] = value
        
    del doc
    gc.collect()
    
    # Check that required aliases are present
    required_aliases = ["Année des revenus", "Revenu fiscal de référence", "Nom du déclarant 1"]
    if not all(alias in data for alias in required_aliases):
        logger.warning(f"Missing required aliases. Found: {list(data.keys())}")
        return None
    
    try:
        # Pass the dictionary using the ALIASES as keys
        return AvisImpositionModel(**data)
    except Exception as e:
        logger.error(f"Failed to create AvisImpositionModel: {e}")
        logger.error(f"Data passed: {json.dumps(data, indent=2)}")
        return None

# The rest of the script remains the same as provided in the previous response, 
# including the removal of the debug return in _worker.



def _get_file_extension(content_type: str) -> str:
    """Determine file extension from content type."""
    if 'pdf' in content_type:
        return '.pdf'
    elif 'jpeg' in content_type or 'jpg' in content_type:
        return '.jpg'
    elif 'png' in content_type:
        return '.png'
    else:
        logger.warning(f"Unknown content type: {content_type}, defaulting to .bin")
        return '.bin'

def upload_to_s3(s3_client, bucket: str, prefix: str, file_id: str, tax_notice: AvisImpositionModel, 
                raw_content: bytes, content_type: str, retries: int = 3, delay: int = 1):
    """
    Upload original file and tax notice data as JSON to S3 in Label Studio task format.
    """
    ext = _get_file_extension(content_type)
    raw_key = f"{prefix}files/{file_id}{ext}"
    json_key = f"{prefix}{file_id}.json"
    
    try:
        # Upload original raw file
        s3_client.put_object(
            Bucket=bucket,
            Key=raw_key,
            Body=raw_content,
            ContentType=content_type
        )
        logger.info(f"Uploaded original file to s3://{bucket}/{raw_key}")
        
        # Create and upload Label Studio task
        pdf_url = f"s3://{bucket}/{raw_key}"
        task_data = create_task(pdf_url=pdf_url, ground_truth=tax_notice.model_dump())
        
        json_data = json.dumps(task_data, ensure_ascii=False, indent=2)
        s3_client.put_object(
            Bucket=bucket,
            Key=json_key,
            Body=json_data.encode('utf-8'),
            ContentType='application/json'
        )
        logger.info(f"Uploaded JSON task to s3://{bucket}/{json_key}")
        
    except ClientError as e:
        error_code = e.response.get('Error', {}).get('Code', 'Unknown')
        logger.error(f"S3 upload failed for {file_id} ({error_code}), attempt {4-retries}/3: {e}")
        
        if retries > 1:
            time.sleep(delay)
            upload_to_s3(s3_client, bucket, prefix, file_id, tax_notice, 
                        raw_content, content_type, retries-1, delay*2)
        else:
            logger.error(f"Failed to upload {file_id} after all retries")
            raise

def _worker(file_id: str, bucket: str, prefix: str) -> Tuple[str, bool, Optional[dict]]:
    """
    Worker function executed in a separate process.
    """
    try:
        # Initialize S3 client in worker process
        config = _get_session_config()
        s3_client = boto3.client(
            's3',
            endpoint_url=config['s3_endpoint'],
            aws_access_key_id=config['s3_access_key'],
            aws_secret_access_key=config['s3_secret_key'],
            region_name=config['s3_region']
        )
        
        # Get image for processing and original content for storage
        img, raw_content, content_type = get_tax_notice_data(file_id)
        
        if not img:
            # print("Skipping non pdf")
            return (file_id, False, None)

        # Extract tax notice from image
        tax_notice = extract_tax_notice(img, timeout=DEFAULT_TIMEOUT)
        
        del img
        gc.collect()
        
        if tax_notice:
            # FIX: The return statement has been removed from here
            # Upload original content and extracted data to S3
            upload_to_s3(s3_client, bucket, prefix, file_id, tax_notice, raw_content, content_type)
            return (file_id, True, tax_notice.model_dump())
        
        logger.warning(f"No valid tax notice extracted from {file_id}")
        return (file_id, False, None)
        
    except Exception as e:
        logger.error(f"Error processing {file_id}: {e}", exc_info=True)
        return (file_id, False, None)

def process_dataset(dataset: List[str], bucket_name: str, s3_prefix: str, 
                   chunk_size: int = CHUNK_SIZE, max_workers: int = MAX_WORKERS):
    """
    Process a dataset of tax notice file IDs in parallel.
    """
    success_count = 0
    total_count = len(dataset)
    
    logger.info(f"Processing {total_count} files in chunks of {chunk_size} with {max_workers} workers")
    
    with tqdm(total=total_count, desc="Processing tax notices", unit="file") as pbar:
        for start_idx in range(0, total_count, chunk_size):
            chunk = dataset[start_idx:start_idx + chunk_size]
            
            with ProcessPoolExecutor(max_workers=max_workers) as exe:
                futures = {exe.submit(_worker, fid, bucket_name, s3_prefix): fid for fid in chunk}
                
                for fut in as_completed(futures):
                    file_id = futures[fut]
                    try:
                        file_id, ok, tax_dict = fut.result()
                    except Exception as e:
                        logger.error(f"Future execution failed for {file_id}: {e}")
                        ok, tax_dict = False, None

                    # Avoid division by zero in success rate calculation
                    rate = f"{100*success_count/pbar.n:.1f}%" if pbar.n > 0 else "0.0%"
                    pbar.set_postfix({"success": "✓" if ok else "✗", "rate": rate})
                    pbar.update(1)

                    if ok and tax_dict:
                        success_count += 1
            
            gc.collect()
    
    return success_count, total_count

# Main execution
if __name__ == "__main__":
    # Load dataset
    dataset = load_tax_notice_dataset()[1600:]
    
    # Get configuration
    config = _get_session_config()
    bucket_name = config['s3_bucket']
    s3_prefix = "avis_imposition/"
    
    # Process dataset
    success_count, total_count = process_dataset(
        dataset=dataset,
        bucket_name=bucket_name,
        s3_prefix=s3_prefix,
        chunk_size=CHUNK_SIZE,
        max_workers=MAX_WORKERS
    )
    
    # Final summary
    success_rate = 100 * success_count / total_count if total_count > 0 else 0
    logger.info(f"✓ Processing complete: {success_count}/{total_count} ({success_rate:.1f}%) successful")